# 01 · Exploratory Data Analysis — CIC-IDS2017
What is in the data, how imbalanced it is, which columns are broken (Inf/NaN), and how redundant the features are.
Run `python -m ml.data.build_dataset --download --build` first.

In [ ]:
import sys, os
from pathlib import Path
ROOT = Path.cwd() if (Path.cwd() / "ml").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT)); os.chdir(ROOT)
import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt, joblib, json
print("project root:", ROOT)

In [ ]:
from ml.data.loader import DatasetLoader
df, meta = DatasetLoader().load_dataset()
print({k: v for k, v in meta.items() if k not in ("class_distribution", "sampling_manifest")})
df.shape

In [ ]:
# Class distribution in the sample vs the full 2.83M-row corpus
man = meta.get("sampling_manifest") or {}
dist = pd.DataFrame({"sample": pd.Series(meta["class_distribution"]), "full_corpus": pd.Series(man.get("full_corpus_class_distribution", {}))}).fillna(0).astype(int)
dist.sort_values("sample", ascending=False)

In [ ]:
ax = dist["sample"].sort_values().plot.barh(figsize=(8, 5), logx=True, color="#0072B2")
ax.set_xlabel("rows (log)"); ax.set_title("Sample class distribution"); plt.tight_layout()

In [ ]:
# Missing / infinite values: only two CICFlowMeter columns are affected
num = df.drop(columns=[meta["label_column"]]).apply(pd.to_numeric, errors="coerce")
bad = (num.isna() | np.isinf(num)).sum()
bad[bad > 0]

In [ ]:
# Constant columns (zero variance) and the most redundant feature pairs
const = [c for c in num.columns if num[c].nunique(dropna=True) <= 1]
print("constant columns:", const)
corr = num.drop(columns=const).sample(min(20000, len(num)), random_state=42).corr().abs()
pairs = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool)).stack().sort_values(ascending=False)
pairs.head(15)

In [ ]:
# Do attacks and benign flows look different? Compare medians of a few flow-shape features
feats = ["Flow Duration", "Total Fwd Packets", "Fwd Packet Length Mean", "Init_Win_bytes_forward", "Flow Packets/s", "SYN Flag Count"]
df["is_attack"] = (df[meta["label_column"]] != "BENIGN").astype(int)
df.groupby("is_attack")[feats].median().T